In [ ]:
# The sample: examples/data/cmsopen_sample.root, committed next to this notebook (no download needed).
# Set CMSOPEN_SAMPLE_FILE to use another copy.
import os
from pathlib import Path

DATA_FILE = Path(os.environ.get("CMSOPEN_SAMPLE_FILE") or Path.cwd() / "data" / "cmsopen_sample.root")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"the sample is missing: {DATA_FILE}. Start Jupyter in examples/ of a clone of the repository, or set "
        "CMSOPEN_SAMPLE_FILE=/absolute/path/to/cmsopen_sample.root"
    )
print(f"Sample: {DATA_FILE} ({DATA_FILE.stat().st_size:,} bytes)")

# Layer 9: the whole analysis on a small real-data sample

`examples/data/cmsopen_sample.root` holds the first events of the development file (CERN Open Data record 31129,
Run2012C DoubleMuParked) with only the branches the analysis reads; `data/SAMPLE.md` records its provenance, size
and checksums. It is small enough to commit, so this notebook runs from a fresh clone with no download.

**The fit.** With a few thousand pairs per channel the notebook fits `voigt_poly`: a Breit-Wigner of the
resonance's natural width (fixed) convolved with one Gaussian resolution, on a quadratic background. That line shape
has a single free width, which a few thousand pairs determine. The frozen default models (two resolution cores,
`dev/model_choice.md`) need far more pairs to determine their second core; on the sample they may fail, and the
last check shows that this is reported as an `AnalysisError`, not hidden.

No physics number is asserted: the checks are structural.

In [ ]:
import hashlib
import json
import re
import tempfile
import time

from IPython.display import Image, display

import cmsopen
from cmsopen.fitting import pdg_mass_width
from cmsopen.pipeline import CHANNELS

WINDOW = {"Z_mumu": 1.0, "Jpsi_mumu": 0.02}  # GeV around the PDG mass: generous, structural


def recorded(key):
    '''A value from the provenance table in data/SAMPLE.md.'''
    m = re.search(rf"^\| {key} \| (.+?) \|$", (DATA_FILE.parent / "SAMPLE.md").read_text(), flags=re.M)
    return m.group(1) if m else ""


print(f"cmsopen imported from: {cmsopen.__file__}")
print(f"recorded: {recorded('events')}; {recorded('size')}")
tmp = Path(tempfile.mkdtemp(prefix="cmsopen-layer9-"))
results = {}

## Z $\to \mu^+\mu^-$

In [ ]:
name = "Z_mumu"
start = time.perf_counter()
results[name] = r = cmsopen.analyze(DATA_FILE, channel=name, model="voigt_poly", stability=False, verbose=False,
                                    outdir=tmp / name)
print(f"[time] {time.perf_counter() - start:.1f} s")
print(r.fit.table())
display(Image(filename=str(tmp / name / "mass_fit.png"), width=640))

## J/$\psi \to \mu^+\mu^-$

In [ ]:
name = "Jpsi_mumu"
start = time.perf_counter()
results[name] = r = cmsopen.analyze(DATA_FILE, channel=name, model="voigt_poly", stability=False, verbose=False,
                                    outdir=tmp / name)
print(f"[time] {time.perf_counter() - start:.1f} s")
print(r.fit.table())
display(Image(filename=str(tmp / name / "mass_fit.png"), width=640))

## Self-checks (structural only)

The sample matches its recorded size and SHA-256; both channels give pairs and a converged fit whose peak lies within
a generous window of the PDG mass (Z: 1 GeV, J/$\psi$: 0.02 GeV); the figures and summaries are written; parallel
processing equals serial; the frozen default either fits or fails with the documented exception.

In [ ]:
class Checks:
    """Collects self-check results and prints one OK/FAIL/SKIP line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail):
        """Print one OK or FAIL line and remember failures."""
        print(f"{'OK' if ok else 'FAIL'}: {name} ({detail})")
        if not ok:
            self.failures.append(name)

    def skip(self, name, reason):
        """Print one SKIP line (a check that cannot run here)."""
        print(f"SKIP: {name} ({reason})")

In [ ]:
checks = Checks()
data = DATA_FILE.read_bytes()
checks.check(
    "the sample matches its recorded size and SHA-256 (data/SAMPLE.md)",
    recorded("size") == f"{len(data):,} bytes" and recorded("SHA-256") == hashlib.sha256(data).hexdigest(),
    f"{len(data):,} bytes, sha256 {hashlib.sha256(data).hexdigest()[:16]}...",
)
for name, r in results.items():
    pdg = pdg_mass_width(CHANNELS[name].resonance)[0]
    m, me = r.fit.peak_position
    checks.check(f"{name}: pairs and a converged voigt_poly fit", r.fit.n_entries > 500,
                 f"{r.fit.n_entries:,} pairs in the fit range, chi2/ndf {r.fit.chi2_ndf:.2f}")
    checks.check(f"{name}: the peak lies within {WINDOW[name]:g} GeV of the PDG mass (a generous window)",
                 abs(m - pdg) < WINDOW[name], f"{m:.4f} +- {me:.4f} GeV, PDG {pdg:.4f} GeV")
    written = {p.name for p in (tmp / name).iterdir()}
    checks.check(f"{name}: figures and summaries are written",
                 {"summary.json", "summary.txt", "mass_fit.png", "cutflow.png"} <= written, ", ".join(sorted(written)))

parallel = cmsopen.analyze(DATA_FILE, channel="Z_mumu", model="voigt_poly", stability=False, verbose=False,
                           n_workers=2, chunk_size=10_000)
checks.check("parallel (n_workers=2, chunks of 10,000 events) equals serial",
             json.dumps(parallel.to_dict(), sort_keys=True) == json.dumps(results["Z_mumu"].to_dict(), sort_keys=True),
             "to_dict() compared")

default = CHANNELS["Jpsi_mumu"].model
try:
    r = cmsopen.analyze(DATA_FILE, channel="Jpsi_mumu", stability=False, verbose=False)
    outcome, ok = f"fitted: {r.fit.model_name}, chi2/ndf {r.fit.chi2_ndf:.2f}", r.fit.model_name == default
except cmsopen.AnalysisError as exc:
    outcome, ok = f"AnalysisError: {str(exc)[:110]}", "fit failed" in str(exc)
print(f"The frozen J/psi default ({default}) on the sample: {outcome}")
checks.check("the frozen default either fits or fails with the documented AnalysisError", ok, outcome[:60])

if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")